PANORAMICA SU TENSORFLOW E PYTORCH: DIFFERENZE CHIAVE

In [ ]:
I TENSORI

I tensori sono come dei mattoncini per costruire qualcosa, mattoncini che sanno fare calcolo a velocità della luce.

- Anatomia del Tensore: differenze con array np
- Navigazione Multidimensionale all'interno del Tensore
- Manipolazione delle forma del tensore

Se NumPy è così potente con il calcolo vettorializzato, perchè serve PyTorch?
Sebbene in Tensori sembrino array multidimensionali simili a quelli di NumPy, i Tensori rappresentano l'unità fondamentale per il Deep Learning moderno. La loro forza risiede nella capacità di interagire con acceleratori hardware.
Numpy è come un ottima officina meccanica dove tutto si fa bene ma a mano, Pytorch invece, progettato da zero per far girare le reti neurali, è come un'officina automatizzata.
PyTorch è stato costruito con un obbiettivo preciso: far girare le reti neurali facendo in modo di non dover aspettare settimane per un risultato.

Comprendere come convertire i dati tra questi due mondi senza sprecare risorse è il primo passo per costruire pipeline di addestramento efficienti.

Caratteristiche di PyTorch
- Device Awareness: a differenza di NumPy, un tensore può risiedere sia nella CPU che sulla GPU, permettendo calcoli massivamente paralleli. Con la GPU posso fare operazioni semplici ma tutte insieme, mentra la CPU è limitata ad una sola operazione per volta.
- Tracking dei Gradienti: i tensori possono incorporare la logica per il calolo automatico delle derivate necessarie per la backpropagation. I tensori hanno una memoria storica
- Zero-copy Bridge: PyTorch permette di creare tensori da array NumPy condividendo lo stesso spazio di memoria sottostante, senza dover duplicare i dati, risparmiando tempo e memoria.
- Tipizzazione Rigida: i tensori richiedono una definizione precisa del tipo di dato per ottimizzare le operazioni sui registri della scheda video.

In informatica muovere i dati è spesso più lento che elaborarli.
Utilizzando torch.from_numpy() creiamo un legame diretto dove la modifica di un elemento nell'array orignale influisce istantaneamente sul tensore.
Il metodo .to() o .cuda() sposta fisicamente i dati dai banchi RAM della CPU alla VRAM della GPU, un'operazione costosa che va limitata e da fare solo se serve veramente.
Funzioni come .torch.randn() o torch.zeros() inizializzano i tensori direttamente nel framework, predisponendo il grafo computazionale, pronti per essere modellati dalle nostre funzioni.

Dietro ogni tensore c'è lo storage,immagina un magazzino lineare dove tutti i numeri sono messi in fila, il tensore è solo un'etichetta che ci dice come leggere quel magazzino.
Quando convertiamo da NumPy a PyTorch ci dice ehi non serve un nuovo magazzino, userò il tuo', questo rende tutto più veloce, ma attenzione, se cambiate il valore nel magazzino, entrambi vedono la modifica.

NAVIGAZIONE ALL'INTERNO DEL TENSORE

Estrarre un riga, una colonna o un sotto-volume da un tensore è un'operazione quotidiana. PyTorch adotta una sintassi ereditata da Python ma estesa a n-dimensioni.
Lo Slicing è lo strumento che ci permette di fare estrazioni. Con pochi simboli diciamo alla macchina quale fetta di dati vogliamo analizzare

ESTRAZIONE
- Basic Slicing: utilizza la notazione 'start:stop:step' per estrarre intervalli contigui lungo una specifica dimensione.
- Advanced Indexing: permete di passare liste o tensori di indici per selezionare elementi non contigui in modo arbitrario: Vuoi elementi sparsi: passa una lista di indici.
- Ellipsis Operator: l'uso di '...' consente di saltare tutte le dimensioni intermedie, rendendo il codice più leggibile e generico. E' come dire: non mi interessa quante dimensioni ci sono in mezzo, vai direttamente all'ultima
- Boolean Masking: selezionare degli elementi che soddisfano una determinata condizione logica.. Passano solo i dati che superano la nostra prova logica.

ACCESSO
- Accesso al singolo elemento: Utilizzando .item() è possibile estrarre un valore scalare da un tensore di dimensione uno per riportarlo nel mondo python standard
- Slicing su Batch: Quando lavoriamo con i Batch (ovvero i gruppi di immagini) In una struttura 'Batch x Canali x Altezza x Larghezza' l'espressione '[:10,0,:,:'] seleziona il primo canale delle prime dieci immagini
- Dimensioni negative: Come nelle liste di Python, l'indice negativo'-1' si riferisce all'ultima dimensione, utile quando non si conosce a priori la profondità del tensore

COMPORTAMENTO DELLA MEMORIA
Lo slicing di base produce sempre una 'view', ovvero un tensore che punto alla stessa memoria dell'originale. Modificare la 'fetta' estratta modificherà anche il tensore sorgente.
L'indexing avanzato, al contrario, produce spesso una copia fisica dei dati (come fare una fotocopia, i dati vengono effettivamente copiati), poichè gli elementi selezionati potrebbero non essere memorizzati in modo contiguo.
E' quindi importante sapere quando si sta utilizzando una vista o una copia dei dati

CAMBIARE LA FORMA DELLA MEMORIA (reshape e transpose)
I dati spesso arrivano in formati non compatibili con i layer della nostra rete. Cambiare la forma di un tensore senza alterare i dati è una competenza fondamentale. I dati sono gli stessi, cambia solo come li disponiamo.

- Metodo view() cambia la forma del tensore senza copiare i dati, ma richiede che il tensore originale sia memorizzato in modo contiguo.
- Metodo reshape(): simile a view, ma più flessibile; se necessario, effettua una copia dei dati per garantire la nuova forma (se i dati sono disordinati).
- Transpose: scambia esattamente due dimensioni tra loro, operazione tipica del calcolo matriciale classico
- Permute: generalizzazione di traspose che permette di riordinare tutte le dimensioni del tensore in un unico passaggio

Il volume totale degli elementi deve rimanere lo stesso, stiamo solo cambiando il contenitore, non devono sparire/comparire dati.

Lavorare con i Tensori 3D
Lavorare in 3D significa gestire Profondità Altezza, Larghezza.
3D significa 3 indici per accedere ai dati, non necessariamente altezza, larghezza, profondità
Un tensore 3D può rappresentare una sequenza temporale o un'immagine RGB.

Utilizzando -1 in view o reshape, pytorch calcola automaticamente la dimensione mancate basandosi sul numero totale di elementi.

Squeeze e Unsqueeze
Operazioni per aggiungere o rimuovere dimensioni 'vuote' di grandezza unitaria, fondamentali per far combaciare i batch

Attenzione perchè ruotare troppo i dati può portare confusione all'interno del computer. Dopo una rotazione o un trasposizione i dati non sono più contigui, non sono più uno difianco all'altro. la soluzione è il metodo .contiguous

Contiguità e Prestazioni
Il vincolo della memoria lineare.
Dopo un operazione di 'transpose' i dati nel computer non sono più ordinati in modo lineare. Chiamare 'view' su un tensore trasposto fallirà a meno di non invocare prima .contiguous()
tensore.transpose(0,1).contiguous().view(-1)
E' come dire alla fabbrica di riorganizzare gli scaffali del magazzino, costa un pò di tempo, ma è necessario per andare avanti.

In [1]:
import torch
import numpy as np

In [ ]:
#1. IL PONTE TRA NUMPY E PYTORCH
#Creiamo un array NumPy classico (CPU-only, calcolo scientifico standard)
np_array=np.array([[1.0,2.0,3.0],[4.0,5.0,6.0]]) #elaborato tramite CPU

#Conversione in Tensore: notate come condividono la memoria
#Se cambiate np_arraty, cambierà anche tensor_from_np
tensor_from_np=torch.from_numpy(np_array) #converto array in tensore, utilizza memoria condivisa tra i due elementi

print(f"Tensore creato da NumPy:\n{tensor_from_np}")
print(f"Device attuale: {tensor_from_np.device} (Sempre CPU per NumPy)")

Tensore creato da NumPy:
tensor([[1., 2., 3.],
        [4., 5., 6.]], dtype=torch.float64)
Device attuale: cpu (Sempre CPU per NumPy)


#2. SLICING E INDEXING (NAVIGAZIONE SPAZIALE)

In [ ]:
#Creiamo un tensore 3D: immagine 2 metrici, ognuna 3x4
#Shape: [pagine, righe, colonne] -> [2,3,4]
t3d=torch.randn(2,3,4)
print(t3d)

tensor([[[ 0.8885,  1.1798,  0.2356,  0.4107],
         [-0.4780,  1.0337,  0.8655,  0.3929],
         [-1.0144,  0.6771, -0.0831, -0.9770]],

        [[-0.7908,  0.0705, -1.0902,  0.9339],
         [ 0.4831, -0.6582, -0.1940, -0.1775],
         [ 0.2672,  0.0369, -0.5868,  0.2964]]])


In [ ]:
# Estrazione: vogliamo tutte le righe, ma solo la seconda colonna, delle prima pagina
#Utiliziamo l'indice 0 per la prima pagina e 1 per la seconda colonna (zero-based)
slice_1=t3d[0,:,1]   #pagina, righe, colonne
print(slice_1)

tensor([1.1798, 1.0337, 0.6771])


In [6]:
#Uso dell'operatore Ellipsis (...) per dire 'prendi tutto il resto'
# Prendo l'ultima colonna di tutte le pagine e tutte le righe
slice_2=t3d[...,-1]
print(slice_2)

tensor([[ 0.4107,  0.3929, -0.9770],
        [ 0.9339, -0.1775,  0.2964]])


In [ ]:
#3. RESHAPE E TRANSPOSE (CAMBIARE LA GEOMETRICA)
#Vogliamo 'appliattire' il nostro tensore 3D in una matrice 2D
#Il numero totale di elementi deve restare costante (2*3*4=24)
#Usiamo -1 per far calcolare a PyTorch la dimensione mancante
t_flat=t3d.view(2,-1) #risultato [2,12]  #con view trasformo il tensore 3d in una matrice 2d, il numero di elementi totali rimane lo stesso
print(t_flat)
#pytorch calcola la dimensione rimanente per non perder ei dati

tensor([[ 0.8885,  1.1798,  0.2356,  0.4107, -0.4780,  1.0337,  0.8655,  0.3929,
         -1.0144,  0.6771, -0.0831, -0.9770],
        [-0.7908,  0.0705, -1.0902,  0.9339,  0.4831, -0.6582, -0.1940, -0.1775,
          0.2672,  0.0369, -0.5868,  0.2964]])


In [ ]:
#Scambio le prime due dimensioni (Transpose)
#Utile se vogliamo passare da [Batch, Canali, Larghezza] a [Canali, Batch, Larghezza]
t_transposed=t3d.transpose(0,1) #risultato [3,2,4)]   da 2,3,4
print(t_transposed)

tensor([[[ 0.8885,  1.1798,  0.2356,  0.4107],
         [-0.7908,  0.0705, -1.0902,  0.9339]],

        [[-0.4780,  1.0337,  0.8655,  0.3929],
         [ 0.4831, -0.6582, -0.1940, -0.1775]],

        [[-1.0144,  0.6771, -0.0831, -0.9770],
         [ 0.2672,  0.0369, -0.5868,  0.2964]]])


I Tensori supearano NumPy grazie all'accelerazione GPU e al calcolo dei gradienti, mantenendo un ponte efficiente tra i due mondi.
Slicing e Indexing permettono di isolare porzioni di dati con logiche contigue o arbitrarie tramite maschere booleane.
Raspape, Permute e Transpose sono strumenti essenziali per adattare la forma dei tensori alla necessità dei layer neurali senza distruggere l'informazione.